In [ ]:
import os
import cv2
import numpy as np
import pickle
from insightface.model_zoo import get_model

In [ ]:
# --- Load model ---
model = get_model("weights/sface.onnx", providers=["CUDAExecutionProvider"])
model.prepare(ctx_id=0)

def get_embedding(img_path):
    try:
        img = cv2.imread(img_path)
        if img is None:
            raise ValueError("Image not loaded")
        # Convert BGR → RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        emb = model.get_feat(img_rgb)
        return emb
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return None

In [ ]:
# --- Build database ---
database_dir = "../new_database"
embeddings = []
ids = []

for person in sorted(os.listdir(database_dir)):
    person_path = os.path.join(database_dir, person)
    if not os.path.isdir(person_path):
        continue

    for file in sorted(os.listdir(person_path)):
        ext = os.path.splitext(file)[1].lower()
        if ext not in {".jpg", ".jpeg", ".png"}:
            continue

        img_path = os.path.join(person_path, file)
        emb = get_embedding(img_path)
        if emb is not None:
            embeddings.append(emb)
            ids.append(person)

# --- Save database ---
os.makedirs("database", exist_ok=True)
np.savez("database/sface_database.npz",
         embeddings=np.vstack(embeddings),
         ids=np.array(ids))

print(f"Saved {len(ids)} embeddings to database/sface_database.npz")